# 04: Datasets, Chat Templates, and Loss Masking

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/04_datasets_chat_templates_and_loss_masking.ipynb)

Finetuning quality is heavily shaped by formatting. The same examples can teach the wrong lesson if roles are wrong, separators are inconsistent, or the loss is applied to tokens you never intended to train on.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Learning goals

- distinguish prompt-completion data from multi-turn conversational data
- understand why chat templates are training decisions, not just display decisions
- build assistant-only and completion-only masking intuition from first principles
- convert between common open-source dataset shapes
- split train and validation sets before training and inspect tokenization carefully

In [ ]:
import random
from collections import Counter
from pathlib import Path

random.seed(2026)

ARTIFACT_DIR = Path("artifacts") / "notebook_04_datasets_templates_masks"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        headers = list(rows[0].keys())

    widths = []
    for header in headers:
        longest = max(len(str(row.get(header, ""))) for row in rows)
        widths.append(max(len(str(header)), longest))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(header, "")).ljust(widths[index]) for index, header in enumerate(headers)))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Why formatting matters so much

Training data teaches a conditional pattern:

given this prefix, continue in that style

If the prefix is malformed, then the learned behavior is malformed too. That is why chat templates and masking belong in the foundations module, not as a later footnote.

In [ ]:
prompt_completion_examples = [
    {
        "instruction": "Rewrite the note in a calmer tone.",
        "input": "Customer is upset about a delayed refund.",
        "output": "I am sorry the refund has taken longer than expected. We are checking the status now.",
    },
    {
        "instruction": "Extract the shipping city.",
        "input": "Ship replacement parts to Austin, Texas by Friday.",
        "output": "Austin",
    },
]

conversation_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a concise support assistant."},
            {"role": "user", "content": "My invoice total looks wrong."},
            {"role": "assistant", "content": "I can help check that. Which line item looks off to you?"},
            {"role": "user", "content": "The tax line seems doubled."},
            {"role": "assistant", "content": "Please share the subtotal and tax line so I can verify the math."},
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Summarize this incident in one sentence."},
            {"role": "assistant", "content": "A database failover caused a 14 minute reporting outage before traffic was restored."},
        ]
    },
]

print("Prompt-completion examples:", len(prompt_completion_examples))
print("Conversation examples:", len(conversation_examples))

## Prompt-completion versus conversational data

Both are valid. They just teach different distributions.

- prompt-completion examples are simple and direct
- conversational examples teach role structure, turn-taking, and assistant behavior across context

Many modern instruct models expect chat-shaped data, even if your original dataset did not start that way.

In [ ]:
format_rows = [
    {
        "format": "prompt_completion",
        "best_for": "single-turn instruction following",
        "shape": "instruction, optional input, output",
        "main_risk": "loses role structure for multi-turn tasks",
    },
    {
        "format": "conversation",
        "best_for": "assistant behavior across turns",
        "shape": "messages list with explicit roles",
        "main_risk": "template mistakes can silently shift supervision",
    },
]

print_table(format_rows, headers=["format", "best_for", "shape", "main_risk"])

## Convert prompt-completion records into message lists

If the target model is chat-native, a useful default is to wrap single-turn examples in a messages structure. That gives you one consistent downstream preprocessing path.

In [ ]:
def prompt_completion_to_messages(example, system_prompt=None):
    user_parts = [example["instruction"].strip()]
    if example.get("input"):
        user_parts.append(example["input"].strip())
    user_text = "\n\n".join(part for part in user_parts if part)

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_text})
    messages.append({"role": "assistant", "content": example["output"].strip()})
    return {"messages": messages}

converted_examples = [
    prompt_completion_to_messages(example, system_prompt="You are a careful enterprise assistant.")
    for example in prompt_completion_examples
]

print(converted_examples[0])

## A template is a contract

Chat models do not train on abstract roles. They train on rendered tokens. So a chat template is a contract between:

- your data shape
- the tokenizer
- the training objective

If any one of those is inconsistent, the model learns the wrong continuation boundary.

In [ ]:
def render_simple_chat(messages):
    lines = []
    for message in messages:
        role = message["role"].upper()
        lines.append("<" + role + ">")
        lines.append(message["content"].strip())
    lines.append("<ASSISTANT>")
    return "\n".join(lines)

sample_render = render_simple_chat(converted_examples[0]["messages"][:-1]) + converted_examples[0]["messages"][-1]["content"]
print(sample_render)

## What loss masking is doing

The loss decides which tokens matter for learning.

- assistant-only loss means the model is trained mainly on assistant responses in a chat transcript
- completion-only loss means the model is trained on the completion part of a prompt-completion example

Both are ways of saying: do not spend gradient on everything equally.

In [ ]:
toy_segments = [
    {"text": "<SYSTEM> You are concise.\n", "train": False},
    {"text": "<USER> Explain QLoRA simply.\n", "train": False},
    {"text": "<ASSISTANT> QLoRA keeps the base model frozen in 4-bit form and trains small adapters.", "train": True},
]

def whitespace_tokenize_with_mask(segments):
    tokens = []
    mask = []
    for segment in segments:
        pieces = segment["text"].split()
        tokens.extend(pieces)
        mask.extend([1 if segment["train"] else 0] * len(pieces))
    return tokens, mask

tokens, mask = whitespace_tokenize_with_mask(toy_segments)
print("Tokens:", tokens)
print("Mask:  ", mask)

## Visualize the mask

A simple visualization makes the concept concrete: only the assistant region contributes training signal here.

In [ ]:
for token, flag in zip(tokens, mask):
    marker = "LEARN" if flag else "skip "
    print(marker, "|", token)

## Split train and validation before training

Validation is not a cleanup step after a run. It is part of the design. If you do not hold out representative examples, you cannot tell whether the adapter generalized or merely copied patterns from the train set.

In [ ]:
all_examples = converted_examples + conversation_examples

def simple_split(examples, validation_fraction=0.25, seed=2026):
    items = list(examples)
    rng = random.Random(seed)
    rng.shuffle(items)
    cut = max(1, int(len(items) * validation_fraction))
    return items[cut:], items[:cut]

train_examples, val_examples = simple_split(all_examples, validation_fraction=0.33)

print("Train size:", len(train_examples))
print("Validation size:", len(val_examples))

## Preserve task diversity across the split

For small notebook datasets, we often do not need full-blown stratification code. But we should still think in those terms:

- task family balance
- turn count balance
- easy vs hard case balance
- edge cases in validation, not only in train

In [ ]:
annotated_examples = [
    {"task_type": "rewriting", "turns": len(converted_examples[0]["messages"]), "source": "pc_1"},
    {"task_type": "extraction", "turns": len(converted_examples[1]["messages"]), "source": "pc_2"},
    {"task_type": "support_dialog", "turns": len(conversation_examples[0]["messages"]), "source": "chat_1"},
    {"task_type": "summarization", "turns": len(conversation_examples[1]["messages"]), "source": "chat_2"},
]

task_counts = Counter(item["task_type"] for item in annotated_examples)
turn_counts = Counter(item["turns"] for item in annotated_examples)

print("Task counts:", dict(task_counts))
print("Turn counts:", dict(turn_counts))

## Build general format conversion helpers

Open-source datasets arrive in many shapes. It is useful to normalize them into one internal representation.

In [ ]:
def messages_to_training_text(messages):
    rendered = []
    for message in messages:
        rendered.append(message["role"].upper() + ": " + message["content"].strip())
    return "\n".join(rendered)

def example_to_sft_record(example):
    if "messages" in example:
        messages = example["messages"]
    else:
        messages = prompt_completion_to_messages(example)["messages"]
    return {
        "messages": messages,
        "text": messages_to_training_text(messages),
    }

sft_records = [example_to_sft_record(item) for item in all_examples]
print(sft_records[0]["text"])

## Common data bugs worth checking early

A small audit catches many expensive mistakes:

- missing assistant turns
- roles in the wrong order
- empty strings
- duplicated examples
- malformed system messages

Do not wait for the trainer to reveal these through weird loss curves.

In [ ]:
problem_records = [
    {"messages": [{"role": "user", "content": "Help"}, {"role": "assistant", "content": ""}]},
    {"messages": [{"role": "assistant", "content": "Hello first"}]},
    {"messages": [{"role": "user", "content": "Repeat"}, {"role": "user", "content": "Still user"}]},
]

def audit_messages(example):
    issues = []
    messages = example.get("messages", [])
    if not messages:
        issues.append("empty_example")
        return issues
    if messages[0]["role"] == "assistant":
        issues.append("starts_with_assistant")
    if messages[-1]["role"] != "assistant":
        issues.append("does_not_end_with_assistant")
    for index, message in enumerate(messages):
        if not message.get("content", "").strip():
            issues.append("empty_content_at_" + str(index))
    consecutive_users = any(
        messages[index]["role"] == messages[index - 1]["role"] == "user"
        for index in range(1, len(messages))
    )
    if consecutive_users:
        issues.append("consecutive_user_turns")
    return issues

for record in problem_records:
    print(record)
    print("Issues:", audit_messages(record))
    print()

## Load the real tokenizer and model stack

The earlier cells used simple toy renderers so the masking logic stayed visible. Now we switch to the shared setup so we can inspect a real tokenizer and a real chat template path.

In [ ]:
# --- Google Colab Pro Fine-Tuning Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Inspect real chat templating

The tokenizer ultimately decides how a message list becomes model input. That is why a correct messages structure is necessary but not sufficient.

In [ ]:
real_messages = [
    {"role": "system", "content": "You are a concise finetuning tutor."},
    {"role": "user", "content": "Explain why assistant-only loss is useful."},
    {"role": "assistant", "content": "It prevents the model from spending learning capacity on copying the prompt side of the conversation."},
]

rendered_prompt = tokenizer.apply_chat_template(
    real_messages,
    tokenize=False,
    add_generation_prompt=False,
)

token_ids = tokenizer(rendered_prompt, add_special_tokens=False)["input_ids"]

print(rendered_prompt[:800])
print()
print("Token count:", len(token_ids))
print("First 30 token ids:", token_ids[:30])

## Build a token-level assistant mask with real tokenization

One simple first-principles approach is to tokenize the conversation segment by segment, then mark which segments belong to assistant responses.

In [ ]:
def tokenize_segments_with_mask(messages):
    token_ids = []
    labels = []
    for message in messages:
        prefix = message["role"].upper() + ": "
        content = message["content"].strip() + "\n"
        segment_text = prefix + content
        segment_ids = tokenizer(segment_text, add_special_tokens=False)["input_ids"]
        token_ids.extend(segment_ids)
        labels.extend([1 if message["role"] == "assistant" else 0] * len(segment_ids))
    return token_ids, labels

segment_token_ids, assistant_mask = tokenize_segments_with_mask(real_messages)

print("Token count:", len(segment_token_ids))
print("Assistant-labeled tokens:", sum(assistant_mask))
print("Mask preview:", assistant_mask[:60])

## Inspect the mask at token granularity

Seeing a few decoded token chunks helps verify that the supervision boundary is where we think it is.

In [ ]:
preview_rows = []
for index, token_id in enumerate(segment_token_ids[:40]):
    preview_rows.append({
        "index": index,
        "token_id": token_id,
        "decoded": tokenizer.decode([token_id]),
        "train_flag": assistant_mask[index],
    })

print_table(preview_rows, headers=["index", "token_id", "decoded", "train_flag"])

## Build a small train and validation DatasetDict

The training stack wants consistent fields. Here we create a minimal dataset with:

- messages for inspection
- text for trainer-friendly formatting
- assistant token counts for debugging

In [ ]:
def record_with_stats(example):
    messages = example_to_sft_record(example)["messages"]
    token_ids, labels = tokenize_segments_with_mask(messages)
    return {
        "messages": messages,
        "text": messages_to_training_text(messages),
        "token_count": len(token_ids),
        "assistant_token_count": int(sum(labels)),
    }

train_records = [record_with_stats(example) for example in train_examples]
val_records = [record_with_stats(example) for example in val_examples]

dataset = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
})

print(dataset)

## Inspect mask statistics across the split

Before training, it is useful to verify that token counts and assistant-token counts look plausible across both splits.

In [ ]:
summary_rows = []
for split_name in ["train", "validation"]:
    split_ds = dataset[split_name]
    token_counts = split_ds["token_count"]
    assistant_counts = split_ds["assistant_token_count"]
    summary_rows.append({
        "split": split_name,
        "examples": len(split_ds),
        "avg_tokens": round(sum(token_counts) / len(token_counts), 1),
        "avg_assistant_tokens": round(sum(assistant_counts) / len(assistant_counts), 1),
    })

print_table(summary_rows, headers=["split", "examples", "avg_tokens", "avg_assistant_tokens"])

## Takeaways

- dataset format is part of the training objective, not a cosmetic choice
- chat templates decide the exact token stream the model learns from
- assistant-only and completion-only masking keep supervision focused
- train and validation splits should exist before the first run
- tokenization inspection catches silent formatting bugs early

You now have the data and environment foundations needed for the first real QLoRA training notebook.